In [ ]:
from fastai.vision.all import *
import re # Needed for the custom labeling function

# --- 1. Load Public Data ---
path = untar_data(URLs.PETS)
path_images = path/'images' 

# Function to extract the label (breed) from the image filename
def get_label(fname):
    return re.search(r'(.+)_\d+\.(jpg|png)$', fname.name).group(1)

# --- 2. DataBlock Setup ---
print('Preparing data pipeline...')
db = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=get_label,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    item_tfms=Resize(460),
    batch_tfms=[*aug_transforms(size=224, min_scale=0.75), Normalize.from_stats(*imagenet_stats)]
)

# --- 3. Create DataLoaders ---
dls = db.dataloaders(path_images, bs=64)

# --- 4. Build Learner ---
learn = vision_learner(
    dls, 
    resnet34, 
    metrics=error_rate
)

# --- 5. Training ---
print('Starting 4 epochs of fine-tuning on the PETS dataset...')
learn.fine_tune(4)
print('Training Complete.')

# --- 6. Evaluation and Interpretation ---
print('\nValidation Results (Loss, Error Rate):')
learn.validate()

interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(10,10))